In [0]:
# Read CSV via Unity Catalog Volume
# First let's check what external locations are available
spark.sql("SHOW EXTERNAL LOCATIONS").show(truncate=False)

In [0]:
spark.sql("SHOW STORAGE CREDENTIALS").show(truncate=False)

In [0]:
# Define storage path
storage_account_name = "shruthistorageaccount"
bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"

# Read CSV from bronze container
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(bronze_path + "credit_card_fraud.csv")

# Verify read
print(f"Row count: {df_raw.count()}")
df_raw.printSchema()
df_raw.show(5)

In [0]:
from pyspark.sql.functions import current_timestamp

# Add ingestion timestamp — only metadata addition allowed in Bronze
df_raw = df_raw.withColumn("ingested_at", current_timestamp())

In [0]:
# Write as Delta table to Unity Catalog bronze schema
df_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fraud_analytics.bronze.credit_card_fraud")

print("Bronze table written successfully!")

In [0]:
# Verify table exists in Unity Catalog
spark.sql("SELECT * FROM fraud_analytics.bronze.credit_card_fraud LIMIT 5").show()

In [0]:
# Check row count and confirm ingested_at column exists
spark.sql("""
    SELECT 
        COUNT(*) as total_rows,
        MIN(ingested_at) as first_ingested,
        MAX(ingested_at) as last_ingested
    FROM fraud_analytics.bronze.credit_card_fraud
""").show()